# ComfyUI + FLUX on Kaggle: generative art engine behind your Cloudflare Tunnel

Runs [ComfyUI](https://github.com/comfyanonymous/ComfyUI) with **FLUX.1** on a free Kaggle GPU and publishes it on your own hostname, behind a password:

| Path | What |
|---|---|
| `/` | The full ComfyUI node editor (load workflows, add LoRAs, inpaint, upscale) |
| `/flux` | A one-box prompt → image page |
| `/flux/api/*` | Small JSON API: `generate`, `jobs`, `workflow`, `models`, `info` |
| `/mcp` | MCP server for coding agents (tools: `generate_image`, `get_job`, `run_workflow`, `list_jobs`, `cancel_job`, `list_models`, `server_status`) |

### Before you run
1. **Accelerator:** GPU T4 x2 (ComfyUI uses one GPU) or P100. **Internet:** on.
2. **A Cloudflare Tunnel** with a public hostname pointing to `http://localhost:7860`.
3. **Kaggle secrets** (Add-ons → Secrets, then tick them for this notebook):
   - `COMFYUI_TUNNEL_TOKEN`: the tunnel token. Without it everything still runs, only without a public URL.
   - `COMFYUI_UI_PASSWORD` (recommended): login password. Browser: any username + this password. Agents: `Authorization: Bearer <password>`. Without it a random password is printed below.
   - `HF_TOKEN` (only for `FLUX_VARIANT = "dev"`): a Hugging Face token whose account accepted the FLUX.1-dev licence.
4. **Run All.** First run: about 10 minutes (install + 17 GB model download). The first image then takes 1–3 minutes while the model loads; after that FLUX.1-schnell takes roughly 30–60 s per 1024² image on a T4.

### Licences
- FLUX.1-**schnell** (default): Apache 2.0, commercial use allowed.
- FLUX.1-**dev**: FLUX.1 [dev] Non-Commercial License. Set `FLUX_VARIANT = "dev"` only if that suits you.
- ComfyUI: GPL-3.0.

Images are saved to `/kaggle/working/comfyui-output` (kept in the notebook's output when you save a version). Logs: `/kaggle/working/logs/`.

In [ ]:
# 1. Settings (edit if you like)
FLUX_VARIANT = "schnell"      # "schnell" (Apache 2.0, 4 steps) or "dev" (non-commercial, ~20 steps, needs HF_TOKEN)
COMFYUI_REF = ""              # "" = newest ComfyUI; or a tag/commit to pin, e.g. "v0.3.60"
EXTRA_COMFY_ARGS = []         # e.g. ["--lowvram"] if you load extra models and run out of GPU memory
PORT = 7860
COMFY_PORT = 8188
COMFY_DIR = "/tmp/ComfyUI"    # code + models live outside /kaggle/working (its 20 GB are for your images)
OUTPUT_DIR = "/kaggle/working/comfyui-output"
LOG_DIR = "/kaggle/working/logs"
APP_DIR = '/kaggle/working/comfy_app'
CHECKPOINTS = {
    "schnell": ("Comfy-Org/flux1-schnell", "flux1-schnell-fp8.safetensors"),
    "dev": ("Comfy-Org/flux1-dev", "flux1-dev-fp8.safetensors"),
}
assert FLUX_VARIANT in CHECKPOINTS, "FLUX_VARIANT must be 'schnell' or 'dev'"
print("Settings saved.")

### 2. App files
These cells only write files into `APP_DIR`.

In [ ]:
# Shared web kit: password, MCP endpoint, proxy, process helpers
# Writes studio_http.py into APP_DIR. Safe to re-run.
import os
APP_DIR = '/kaggle/working/comfy_app'
os.makedirs(APP_DIR, exist_ok=True)
with open(os.path.join(APP_DIR, 'studio_http.py'), "w", encoding="utf-8") as f:
    f.write(r'''"""studio_http: the small web kit behind the newer agentic-notebooks apps.

build_notebook.py copies this file into each notebook's app folder, so every notebook stays
self-contained: nothing is imported from GitHub at run time. Standard library only.

An app built on it gets:
- routes with regex paths, JSON in and out, file (with byte ranges), zip and chunked responses;
- one password for everything: browsers use HTTP Basic (any username), agents send
  ``Authorization: Bearer <password>`` or ``X-Access-Token: <password>``. ``GET /health`` stays open;
- an MCP server at ``POST /mcp`` (Streamable HTTP transport, plain JSON responses, no sessions),
  and the same tools as REST: ``GET /mcp/tools`` and ``POST /mcp/tools/<name>`` with the arguments as JSON;
- an optional reverse proxy (HTTP and WebSocket) for every path no route matches;
- IPv4 and IPv6 listeners on one port (cloudflared may dial [::1] for "localhost").

The notebook cells use the helpers at the bottom: kaggle_secret(), spawn() / stop_process() with pid
files (so re-running a cell, or the whole notebook after a kernel restart, never leaves a second copy
running), wait_http(), tail(), start_tunnel() and keep_alive(). Logs go to files, never to the notebook.
"""
import base64
import hmac
import http.client
import json
import logging
import mimetypes
import os
import re
import select
import signal
import socket
import subprocess
import sys
import threading
import time
import traceback
import urllib.error
import urllib.request
import zipfile
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from urllib.parse import parse_qs, quote, urlparse

MCP_VERSIONS = ("2025-06-18", "2025-03-26", "2024-11-05")
HOP = {"connection", "keep-alive", "proxy-authenticate", "proxy-authorization", "te", "trailer", "trailers",
       "transfer-encoding", "upgrade"}
SECRET_HEADERS = {"authorization", "x-access-token"}
PID_DIR = "/kaggle/working/.pids" if os.path.isdir("/kaggle/working") else "/tmp/agentic-notebooks-pids"


# ====================================================================== responses
class HTTPError(Exception):
    def __init__(self, status, message):
        super().__init__(message)
        self.status = status


class Response:
    def __init__(self, body=b"", status=200, ctype="text/plain; charset=utf-8", headers=None):
        self.body = body.encode("utf-8") if isinstance(body, str) else body
        self.status, self.ctype, self.headers = status, ctype, dict(headers or {})


class FileResponse:
    """A file from disk. ``name`` makes it a download. Single byte ranges work (audio/video seeking)."""

    def __init__(self, path, name=None, ctype=None, cache="no-cache"):
        if not os.path.isfile(path):
            raise HTTPError(404, "file not found")
        self.path, self.name, self.cache = path, name, cache
        self.ctype = ctype or mimetypes.guess_type(path)[0] or "application/octet-stream"


class StreamResponse:
    """Chunked response from an iterable of bytes or str."""

    def __init__(self, chunks, ctype="text/plain; charset=utf-8", headers=None):
        self.chunks, self.ctype, self.headers = chunks, ctype, dict(headers or {})


class ZipResponse:
    """Streams ``folder`` (or a list of (path, arcname) pairs) as a zip download."""

    def __init__(self, files, name):
        if isinstance(files, str):
            root = os.path.dirname(os.path.abspath(files))
            pairs = []
            for dp, dn, fn in os.walk(files):
                dn.sort()
                for f in sorted(fn):
                    if not f.endswith((".tmp", ".part")):
                        pairs.append((os.path.join(dp, f), os.path.relpath(os.path.join(dp, f), root)))
            files = pairs
        self.files, self.name = files, name


class ToolContent(list):
    """Return this from an MCP tool to send raw content blocks, for example an image."""


def text_block(text):
    return {"type": "text", "text": text}


def image_block(data, mime="image/png"):
    return {"type": "image", "data": base64.b64encode(data).decode("ascii"), "mimeType": mime}


def file_logger(path, name="app"):
    """A logger that writes to ``path`` only (nothing reaches the notebook output)."""
    log = logging.getLogger(name)
    path = os.path.abspath(path)
    if not any(getattr(h, "baseFilename", None) == path for h in log.handlers):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        h = logging.FileHandler(path, encoding="utf-8")
        h.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(message)s", "%Y-%m-%d %H:%M:%S"))
        log.addHandler(h)
    log.setLevel(logging.INFO)
    log.propagate = False
    return log


def _disposition(name):
    ascii_name = re.sub(r'[^A-Za-z0-9_.() -]+', "_", name) or "download"
    return "attachment; filename=\"%s\"; filename*=UTF-8''%s" % (ascii_name, quote(name))


# ====================================================================== requests
class Request:
    def __init__(self, handler, params, max_body):
        self.handler = handler
        u = urlparse(handler.path)
        self.method = handler.command
        self.path = u.path
        self.query = {k: v[-1] for k, v in parse_qs(u.query, keep_blank_values=True).items()}
        self.params = params
        self.headers = handler.headers
        self.max_body = max_body
        self.consumed = False

    @property
    def length(self):
        try:
            return int(self.headers.get("Content-Length") or 0)
        except ValueError:
            raise HTTPError(400, "bad Content-Length")

    @property
    def chunked(self):
        return "chunked" in (self.headers.get("Transfer-Encoding") or "").lower()

    def has_body(self):
        return self.chunked or self.length > 0

    def iter_body(self, limit=None):
        """Yields the request body in pieces (Content-Length or chunked), enforcing ``limit`` bytes."""
        limit = self.max_body if limit is None else limit
        too_big = HTTPError(413, "request body too large (limit %d MB)" % (limit // 2 ** 20))
        rf = self.handler.rfile
        self.consumed = True
        if self.chunked:
            total = 0
            while True:
                line = rf.readline(1024)
                try:
                    size = int(line.split(b";")[0].strip() or b"0", 16)
                except ValueError:
                    raise HTTPError(400, "bad chunked body")
                if size == 0:
                    while rf.readline(1024) not in (b"\r\n", b"\n", b""):
                        pass
                    return
                total += size
                if total > limit:
                    raise too_big
                left = size
                while left:
                    data = rf.read(min(left, 1 << 20))
                    if not data:
                        raise HTTPError(400, "request body was cut off")
                    left -= len(data)
                    yield data
                rf.readline(8)
        else:
            left = self.length
            if left > limit:
                raise too_big
            while left:
                data = rf.read(min(left, 1 << 20))
                if not data:
                    raise HTTPError(400, "request body was cut off")
                left -= len(data)
                yield data

    def body(self):
        return b"".join(self.iter_body())

    def json(self):
        raw = self.body()
        if not raw.strip():
            return {}
        try:
            return json.loads(raw.decode("utf-8"))
        except ValueError:
            raise HTTPError(400, "body is not valid JSON")

    def save_body(self, path, limit):
        """Writes the raw body to ``path`` (atomically) and returns its size."""
        if not self.has_body():
            raise HTTPError(411, "send the file as the raw request body")
        tmp = path + ".part"
        size = 0
        try:
            with open(tmp, "wb") as f:
                for data in self.iter_body(limit):
                    f.write(data)
                    size += len(data)
        except BaseException:
            if os.path.exists(tmp):
                os.remove(tmp)
            raise
        os.replace(tmp, path)
        return size

    def arg(self, name, default=None, cast=str):
        v = self.query.get(name)
        if v in (None, ""):
            return default
        try:
            return cast(v)
        except (TypeError, ValueError):
            raise HTTPError(400, "bad value for %s" % name)


# ====================================================================== app
def _rpc_error(mid, code, message):
    return {"jsonrpc": "2.0", "id": mid, "error": {"code": code, "message": message}}


class App:
    def __init__(self, name, password=None, version="1.0", instructions="", proxy=None, log=None,
                 max_body=64 << 20):
        self.name, self.version, self.instructions = name, version, instructions
        self.password = password or None
        self.proxy = proxy.rstrip("/") if proxy else None
        self.log = log or logging.getLogger(name)
        self.max_body = max_body
        self.routes, self.tools, self.servers = [], {}, []

        self.route("GET", "/health", auth=False)(lambda req: {"ok": True})
        self.route("POST", "/mcp")(self._mcp_http)
        self.route("GET", "/mcp")(lambda req: Response(
            "This MCP server answers POST requests (Streamable HTTP transport, JSON responses).", 405,
            headers={"Allow": "POST"}))
        self.route("DELETE", "/mcp")(lambda req: Response(b"", 405, headers={"Allow": "POST"}))
        self.route("GET", "/mcp/tools")(lambda req: {"tools": self.tool_list()})
        self.route("POST", r"/mcp/tools/(?P<name>[A-Za-z0-9_\-]+)")(self._tool_http)

    # ------------------------------------------------------------------ registration
    def route(self, method, pattern, auth=True):
        rx = re.compile(pattern)

        def deco(fn):
            self.routes.append((method.upper(), rx, fn, auth))
            return fn
        return deco

    def page(self, path, file):
        self.route("GET", re.escape(path))(lambda req: FileResponse(file, ctype="text/html; charset=utf-8"))

    def static(self, prefix, folder):
        root = os.path.realpath(folder)

        def serve(req):
            full = os.path.realpath(os.path.join(root, req.params["rel"]))
            if not full.startswith(root + os.sep):
                raise HTTPError(404, "not found")
            return FileResponse(full)
        self.route("GET", re.escape(prefix) + r"(?P<rel>.+)")(serve)

    def tool(self, name, description, properties=None, required=()):
        """Registers an MCP tool. ``properties`` is a JSON Schema properties object."""
        def deco(fn):
            self.tools[name] = {"fn": fn, "schema": {
                "name": name, "description": description,
                "inputSchema": {"type": "object", "properties": properties or {}, "required": list(required)}}}
            return fn
        return deco

    def tool_list(self):
        return [t["schema"] for t in self.tools.values()]

    def call_tool(self, name, args):
        """Runs a tool. Returns (content_blocks, is_error)."""
        t = self.tools[name]
        if not isinstance(args, dict):
            return [text_block("arguments must be a JSON object")], True
        try:
            out = t["fn"](**args)
        except TypeError as e:
            if "argument" in str(e):
                return [text_block("bad arguments for %s: %s" % (name, e))], True
            self.log.error("tool %s failed\n%s", name, traceback.format_exc())
            return [text_block("TypeError: %s" % e)], True
        except HTTPError as e:
            return [text_block(str(e))], True
        except (ValueError, KeyError) as e:
            return [text_block(str(e).strip("'\""))], True
        except Exception as e:
            self.log.error("tool %s failed\n%s", name, traceback.format_exc())
            return [text_block("%s: %s" % (type(e).__name__, e))], True
        if isinstance(out, ToolContent):
            return list(out), False
        if isinstance(out, str):
            return [text_block(out)], False
        return [text_block(json.dumps(out, indent=1, ensure_ascii=False, default=str))], False

    # ------------------------------------------------------------------ MCP
    def _mcp_http(self, req):
        try:
            msg = json.loads(req.body().decode("utf-8") or "null")
        except ValueError:
            return _rpc_error(None, -32700, "parse error")
        if isinstance(msg, list):
            out = [r for r in map(self._rpc, msg) if r is not None]
            return out if out else Response(b"", 202)
        r = self._rpc(msg)
        return r if r is not None else Response(b"", 202)

    def _rpc(self, m):
        if not isinstance(m, dict) or m.get("jsonrpc") != "2.0":
            return _rpc_error(m.get("id") if isinstance(m, dict) else None, -32600, "invalid request")
        if "method" not in m or "id" not in m:
            return None            # a notification, or a response to a request we never send
        mid, method, params = m["id"], m["method"], m.get("params") or {}
        try:
            if method == "initialize":
                v = params.get("protocolVersion")
                result = {"protocolVersion": v if v in MCP_VERSIONS else MCP_VERSIONS[0],
                          "capabilities": {"tools": {"listChanged": False}},
                          "serverInfo": {"name": self.name, "version": self.version}}
                if self.instructions:
                    result["instructions"] = self.instructions
            elif method == "ping":
                result = {}
            elif method == "tools/list":
                result = {"tools": self.tool_list()}
            elif method == "tools/call":
                name = params.get("name")
                if name not in self.tools:
                    return _rpc_error(mid, -32602, "unknown tool: %s" % name)
                content, err = self.call_tool(name, params.get("arguments") or {})
                result = {"content": content, "isError": err}
            elif method == "resources/list":
                result = {"resources": []}
            elif method == "resources/templates/list":
                result = {"resourceTemplates": []}
            elif method == "prompts/list":
                result = {"prompts": []}
            else:
                return _rpc_error(mid, -32601, "method not found: %s" % method)
        except Exception as e:
            self.log.error("mcp %s failed\n%s", method, traceback.format_exc())
            return _rpc_error(mid, -32603, str(e))
        return {"jsonrpc": "2.0", "id": mid, "result": result}

    def _tool_http(self, req):
        name = req.params["name"]
        if name not in self.tools:
            raise HTTPError(404, "unknown tool: %s" % name)
        content, err = self.call_tool(name, req.json())
        if len(content) == 1 and content[0]["type"] == "text":
            try:
                data = json.loads(content[0]["text"])
            except ValueError:
                data = content[0]["text"]
            body = {"error": data} if err else data
        else:
            body = {"content": content, "isError": err}
        return Response(json.dumps(body, ensure_ascii=False, default=str), 400 if err else 200,
                        "application/json; charset=utf-8")

    # ------------------------------------------------------------------ auth
    def authorized(self, headers):
        pw = self.password
        if not pw:
            return True

        def ok(given):
            return bool(given) and hmac.compare_digest(given.encode("utf-8"), pw.encode("utf-8"))
        if ok(headers.get("X-Access-Token", "")):
            return True
        h = headers.get("Authorization", "")
        if h[:6].lower() == "basic ":
            try:
                given = base64.b64decode(h[6:].strip()).decode("utf-8").partition(":")[2]
            except Exception:
                given = ""
            return ok(given)
        if h[:7].lower() == "bearer ":
            return ok(h[7:].strip())
        return False

    def _deny(self, h):
        self._respond(h, Response(json.dumps({"error": "login required: HTTP Basic (any username) or "
                                              "'Authorization: Bearer <password>'"}), 401,
                                  "application/json; charset=utf-8",
                                  {"WWW-Authenticate": 'Basic realm="%s", charset="UTF-8"' % self.name}))

    # ------------------------------------------------------------------ dispatch
    def dispatch(self, h):
        h._sent = False
        req = None
        try:
            path = urlparse(h.path).path
            method = "GET" if h.command == "HEAD" else h.command
            found, path_known = None, False
            for m, rx, fn, auth in self.routes:
                mt = rx.fullmatch(path)
                if mt:
                    path_known = True
                    if m == method:
                        found = (fn, auth, mt.groupdict())
                        break
            if found is None:
                if path_known:
                    raise HTTPError(405, "method not allowed")
                if not self.proxy:
                    raise HTTPError(404, "not found")
                if not self.authorized(h.headers):
                    return self._deny(h)
                return self._proxy(h)
            fn, auth, params = found
            if auth and not self.authorized(h.headers):
                return self._deny(h)
            req = Request(h, params, self.max_body)
            self._respond(h, fn(req))
        except (BrokenPipeError, ConnectionResetError):
            h.close_connection = True
        except HTTPError as e:
            self._fail(h, e.status, str(e))
        except (ValueError, KeyError) as e:
            msg = ("missing field %s" % e) if isinstance(e, KeyError) else str(e)
            self._fail(h, 400, msg)
        except Exception as e:
            self.log.error("%s %s failed\n%s", h.command, h.path, traceback.format_exc())
            self._fail(h, 500, "%s: %s" % (type(e).__name__, e))
        finally:
            body_left = (req is None or not req.consumed) and (
                "chunked" in (h.headers.get("Transfer-Encoding") or "").lower()
                or (h.headers.get("Content-Length") or "0").strip() not in ("", "0"))
            if body_left:
                h.close_connection = True   # an unread body would corrupt the next request on this connection

    def _fail(self, h, status, message):
        if h._sent:
            h.close_connection = True
            return
        try:
            self._respond(h, Response(json.dumps({"error": message}), status, "application/json; charset=utf-8"))
        except OSError:
            h.close_connection = True

    def _start(self, h, status, headers):
        h.send_response(status)
        h._sent = True
        for k, v in headers:
            h.send_header(k, v)
        h.end_headers()

    def _respond(self, h, out):
        head = h.command == "HEAD"
        if out is None:
            out = Response(b"", 204)
        elif isinstance(out, (dict, list)):
            out = Response(json.dumps(out, ensure_ascii=False, default=str), 200, "application/json; charset=utf-8")
        elif isinstance(out, str):
            out = Response(out)
        elif isinstance(out, bytes):
            out = Response(out, 200, "application/octet-stream")

        if isinstance(out, Response):
            hdrs = {"Content-Type": out.ctype, "Cache-Control": "no-store"}
            hdrs.update(out.headers)
            if out.status not in (204, 304):
                hdrs["Content-Length"] = str(len(out.body))
            self._start(h, out.status, hdrs.items())
            if not head and out.status not in (204, 304) and out.body:
                h.wfile.write(out.body)
        elif isinstance(out, FileResponse):
            self._send_file(h, out, head)
        elif isinstance(out, StreamResponse):
            self._start(h, 200, list({"Content-Type": out.ctype, "Cache-Control": "no-cache",
                                      "X-Accel-Buffering": "no", **out.headers}.items())
                        + [("Transfer-Encoding", "chunked")])
            if head:
                h.wfile.write(b"0\r\n\r\n")
                return
            for chunk in out.chunks:
                if isinstance(chunk, str):
                    chunk = chunk.encode("utf-8")
                if chunk:
                    h.wfile.write(b"%x\r\n" % len(chunk) + chunk + b"\r\n")
                    h.wfile.flush()
            h.wfile.write(b"0\r\n\r\n")
        elif isinstance(out, ZipResponse):
            self._send_zip(h, out, head)
        else:
            raise TypeError("handler returned %r" % type(out))

    def _send_file(self, h, f, head):
        size = os.path.getsize(f.path)
        start, end, status = 0, size - 1, 200
        m = re.fullmatch(r"bytes=(\d*)-(\d*)", (h.headers.get("Range") or "").strip())
        if m and size and (m.group(1) or m.group(2)):
            a, b = m.groups()
            if a == "":
                start = max(0, size - int(b))
            else:
                start, end = int(a), min(int(b), size - 1) if b else size - 1
            if start > end or start >= size:
                return self._start(h, 416, [("Content-Range", "bytes */%d" % size), ("Content-Length", "0")])
            status = 206
        hdrs = [("Content-Type", f.ctype), ("Content-Length", str(end - start + 1)), ("Accept-Ranges", "bytes"),
                ("Cache-Control", f.cache)]
        if status == 206:
            hdrs.append(("Content-Range", "bytes %d-%d/%d" % (start, end, size)))
        if f.name:
            hdrs.append(("Content-Disposition", _disposition(f.name)))
        self._start(h, status, hdrs)
        if head:
            return
        with open(f.path, "rb") as fh:
            fh.seek(start)
            left = end - start + 1
            while left > 0:
                data = fh.read(min(left, 1 << 20))
                if not data:
                    break
                h.wfile.write(data)
                left -= len(data)

    def _send_zip(self, h, z, head):
        self._start(h, 200, [("Content-Type", "application/zip"), ("Content-Disposition", _disposition(z.name)),
                             ("Cache-Control", "no-store"), ("Transfer-Encoding", "chunked")])
        w = h.wfile

        class Chunks:
            def write(self, data):
                if data:
                    w.write(b"%x\r\n" % len(data) + bytes(data) + b"\r\n")
                return len(data)

            def flush(self):
                w.flush()

        if not head:
            with zipfile.ZipFile(Chunks(), "w", zipfile.ZIP_DEFLATED, allowZip64=True) as zf:
                for path, arc in z.files:
                    if isinstance(path, bytes):
                        zf.writestr(arc, path)
                    elif os.path.isfile(path):
                        zf.write(path, arc, compress_type=zipfile.ZIP_STORED if path.endswith(
                            (".safetensors", ".gguf", ".bin", ".npy", ".png", ".jpg", ".mp4", ".zip", ".wav"))
                            else zipfile.ZIP_DEFLATED)
        w.write(b"0\r\n\r\n")
        w.flush()

    # ------------------------------------------------------------------ reverse proxy
    def _proxy(self, h):
        pu = urlparse(self.proxy)
        host, port = pu.hostname, pu.port or 80
        if ("upgrade" in (h.headers.get("Connection") or "").lower()
                and (h.headers.get("Upgrade") or "").lower() == "websocket"):
            return self._proxy_ws(h, host, port)
        req = Request(h, {}, self.max_body)
        headers = [(k, v) for k, v in h.headers.items()
                   if k.lower() not in HOP and k.lower() not in SECRET_HEADERS and k.lower() != "content-length"]
        headers.append(("X-Forwarded-For", h.client_address[0]))
        body = req.body() if req.has_body() else None
        if body is not None:
            headers.append(("Content-Length", str(len(body))))
        conn = http.client.HTTPConnection(host, port, timeout=900)
        try:
            # the original Host header is kept: some backends (ComfyUI) compare it with Origin
            conn.putrequest(h.command, h.path, skip_host="host" in {k.lower() for k, v in headers},
                            skip_accept_encoding=True)
            for k, v in headers:
                conn.putheader(k, v)
            conn.endheaders(body)
            r = conn.getresponse()
        except OSError as e:
            conn.close()
            raise HTTPError(502, "backend not reachable (%s)" % type(e).__name__)
        try:
            hdrs = [(k, v) for k, v in r.getheaders() if k.lower() not in HOP and k.lower() != "content-length"]
            cl = r.getheader("Content-Length")
            chunked = "chunked" in (r.getheader("Transfer-Encoding") or "").lower()
            if h.command == "HEAD" or r.status in (204, 304) or r.status < 200:
                return self._start(h, r.status, hdrs + ([("Content-Length", cl)] if cl else []))
            if cl is not None and not chunked:
                self._start(h, r.status, hdrs + [("Content-Length", cl)])
                while True:
                    data = r.read(1 << 16)
                    if not data:
                        break
                    h.wfile.write(data)
            else:
                self._start(h, r.status, hdrs + [("Transfer-Encoding", "chunked")])
                while True:
                    data = r.read1(1 << 16)
                    if not data:
                        break
                    h.wfile.write(b"%x\r\n" % len(data) + data + b"\r\n")
                    h.wfile.flush()
                h.wfile.write(b"0\r\n\r\n")
        finally:
            conn.close()

    def _proxy_ws(self, h, host, port):
        try:
            up = socket.create_connection((host, port), timeout=10)
        except OSError as e:
            raise HTTPError(502, "backend not reachable (%s)" % type(e).__name__)
        lines = ["%s %s HTTP/1.1" % (h.command, h.path)]
        lines += ["%s: %s" % (k, v) for k, v in h.headers.items() if k.lower() not in SECRET_HEADERS]
        h._sent = True
        h.close_connection = True
        client = h.connection
        try:
            up.sendall(("\r\n".join(lines) + "\r\n\r\n").encode("latin-1"))
            up.settimeout(None)
            client.settimeout(None)
            while True:
                ready, _, broken = select.select([client, up], [], [client, up], 600)
                if broken:
                    break
                for s in ready:
                    data = s.recv(1 << 16)
                    if not data:
                        return
                    (up if s is client else client).sendall(data)
        except OSError:
            pass
        finally:
            up.close()

    # ------------------------------------------------------------------ serving
    def handler_class(self):
        app = self

        class Handler(BaseHTTPRequestHandler):
            protocol_version = "HTTP/1.1"
            server_version = re.sub(r"\W+", "", app.name) or "studio"

            def log_message(self, fmt, *args):
                pass

            def do_GET(self):
                app.dispatch(self)

            do_POST = do_PUT = do_PATCH = do_DELETE = do_HEAD = do_OPTIONS = do_GET
        return Handler

    def start(self, port):
        """Listens on 0.0.0.0 and [::] in background threads. Calling it again restarts cleanly."""
        self.stop()
        handler = self.handler_class()
        for cls, host in ((_V4Server, "0.0.0.0"), (_V6Server, "::")):
            try:
                srv = cls((host, port), handler)
            except OSError as e:
                self.log.warning("could not listen on [%s]:%d: %s", host, port, e)
                continue
            threading.Thread(target=srv.serve_forever, daemon=True).start()
            self.servers.append(srv)
        if not self.servers:
            raise RuntimeError("nothing is listening on port %d" % port)
        self.log.info("%s listening on port %d", self.name, port)

    def stop(self):
        while self.servers:
            srv = self.servers.pop()
            try:
                srv.shutdown()
                srv.server_close()
            except Exception:
                pass

    def serve_forever(self, port):
        """For app processes started with spawn(): serve until SIGTERM."""
        done = threading.Event()
        signal.signal(signal.SIGTERM, lambda *a: done.set())
        signal.signal(signal.SIGINT, lambda *a: done.set())
        self.start(port)
        done.wait()
        self.log.info("stopping")
        self.stop()


class _V4Server(ThreadingHTTPServer):
    allow_reuse_address = True
    daemon_threads = True
    request_queue_size = 64


class _V6Server(_V4Server):
    address_family = socket.AF_INET6

    def server_bind(self):
        self.socket.setsockopt(socket.IPPROTO_IPV6, socket.IPV6_V6ONLY, 1)
        super().server_bind()


# ====================================================================== JSON files and jobs
def read_json(path, default=None):
    try:
        with open(path, encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return default


def write_json(path, data):
    """Atomic write, so readers in other threads or processes never see half a file."""
    tmp = "%s.%d.%d.tmp" % (path, os.getpid(), threading.get_ident())
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=1, ensure_ascii=False, default=str)
    os.replace(tmp, path)


def safe_name(s, n=60):
    return re.sub(r"[^A-Za-z0-9_.-]+", "-", str(s or "")).strip("-.")[:n]


def gpu_info():
    try:
        out = subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.used,memory.total,utilization.gpu",
                              "--format=csv,noheader,nounits"], capture_output=True, text=True, timeout=5).stdout
        gpus = []
        for line in out.strip().splitlines():
            i, n, u, t, g = [x.strip() for x in line.split(",")]
            gpus.append({"index": int(i), "name": n, "mem_used_mb": int(u), "mem_total_mb": int(t), "util": int(g)})
        return gpus
    except Exception:
        return []


# ====================================================================== notebook helpers
def kaggle_secret(name):
    """Value of a Kaggle secret, or None when it is missing or not attached to this notebook.
    Outside Kaggle (local runs, Docker) it reads the environment variable of the same name."""
    try:
        from kaggle_secrets import UserSecretsClient
    except ImportError:
        return os.environ.get(name) or None
    try:
        return UserSecretsClient().get_secret(name) or None
    except Exception:
        return None


def ui_password(secret_name):
    pw = kaggle_secret(secret_name)
    if pw:
        print("UI password: from the %s secret. Browser login: any username + that password." % secret_name)
        return pw
    import secrets
    pw = secrets.token_urlsafe(9)
    print("No %s secret, so this session's password is: %s" % (secret_name, pw))
    print("Browser login: any username + that password. Add the secret to keep a fixed password.")
    return pw


def _pidfile(name):
    return os.path.join(PID_DIR, safe_name(name) + ".json")


def _alive(pid, exe=None):
    try:
        with open("/proc/%d/stat" % pid) as f:
            if f.read().rsplit(")", 1)[1].split()[0] == "Z":
                return False
        if exe:
            with open("/proc/%d/cmdline" % pid, "rb") as f:
                return exe.encode() in f.read()
        return True
    except (OSError, IndexError):
        return False


def process_alive(name):
    info = read_json(_pidfile(name))
    return bool(info) and _alive(info["pid"], info.get("exe"))


def stop_process(name, timeout=20):
    """Stops a process spawn() started under ``name`` (and its children). Returns True if one was running."""
    info = read_json(_pidfile(name))
    if not info:
        return False
    pid, exe = info["pid"], info.get("exe")
    running = _alive(pid, exe)
    if running:
        for sig, wait in ((signal.SIGTERM, timeout), (signal.SIGKILL, 5)):
            try:
                os.killpg(pid, sig)
            except OSError:
                break
            t = time.time()
            while time.time() - t < wait and _alive(pid, exe):
                time.sleep(0.2)
            if not _alive(pid, exe):
                break
    try:
        os.remove(_pidfile(name))
    except OSError:
        pass
    return running


def spawn(name, cmd, log_path, env=None, cwd=None):
    """Starts ``cmd`` in the background with its output appended to ``log_path``.
    A process started earlier under the same name is stopped first, so re-running a cell is safe."""
    stop_process(name)
    os.makedirs(PID_DIR, exist_ok=True)
    os.makedirs(os.path.dirname(os.path.abspath(log_path)), exist_ok=True)
    with open(log_path, "ab") as log:
        log.write(("\n===== %s: starting %s =====\n" % (time.strftime("%Y-%m-%d %H:%M:%S"), name)).encode())
        log.flush()
        proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL,
                                env=env, cwd=cwd, start_new_session=True)
    write_json(_pidfile(name), {"pid": proc.pid, "exe": os.path.basename(cmd[0]), "log": log_path,
                                "started": time.time()})
    return proc


def wait_http(url, timeout=120, name=None, headers=None):
    """Waits until ``url`` answers (any HTTP status). Returns False on timeout or if process ``name`` died."""
    t = time.time()
    while time.time() - t < timeout:
        try:
            urllib.request.urlopen(urllib.request.Request(url, headers=headers or {}), timeout=3)
            return True
        except urllib.error.HTTPError:
            return True
        except Exception:
            pass
        if name and not process_alive(name):
            return False
        time.sleep(1)
    return False


def tail(path, n=30):
    try:
        with open(path, "rb") as f:
            f.seek(0, 2)
            f.seek(max(0, f.tell() - 64 * 1024))
            return "\n".join(f.read().decode("utf-8", "replace").splitlines()[-n:])
    except OSError:
        return "(no log yet at %s)" % path


def ensure_cloudflared(path="/kaggle/working/cloudflared"):
    if not os.path.exists(path):
        url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
        urllib.request.urlretrieve(url, path + ".part")
        os.replace(path + ".part", path)
        os.chmod(path, 0o755)
    return path


def start_tunnel(secret_name, port=7860, log_path="/kaggle/working/logs/cloudflared.log",
                 binary="/kaggle/working/cloudflared"):
    """Starts cloudflared with the token from ``secret_name`` (passed as TUNNEL_TOKEN, never printed).
    A missing secret is not an error: the app keeps running inside the session, without a public URL."""
    stop_process("cloudflared")
    token = kaggle_secret(secret_name)
    if not token:
        print("No public URL: there is no Kaggle secret named %s attached to this notebook.\n"
              "The app is running inside this session anyway. To publish it:\n"
              "  1. Cloudflare Zero Trust -> Networks -> Tunnels: create a tunnel and add a public hostname\n"
              "     pointing to http://localhost:%d\n"
              "  2. Kaggle: Add-ons -> Secrets -> add %s with the tunnel token, and tick it for this notebook\n"
              "  3. Re-run this cell." % (secret_name, port, secret_name))
        return None
    ensure_cloudflared(binary)
    offset = os.path.getsize(log_path) if os.path.exists(log_path) else 0
    proc = spawn("cloudflared", [binary, "tunnel", "--no-autoupdate", "run"], log_path,
                 env=dict(os.environ, TUNNEL_TOKEN=token))

    def new_log():
        with open(log_path, "rb") as f:
            f.seek(offset)
            return f.read().decode("utf-8", "replace")
    for _ in range(45):
        time.sleep(1)
        if "Registered tunnel connection" in new_log():
            print("Tunnel connected. Open the public hostname you routed to http://localhost:%d" % port)
            return proc
        if proc.poll() is not None:
            print("cloudflared stopped (exit code %s). Last log lines:" % proc.returncode)
            print("\n".join(new_log().splitlines()[-15:]))
            print("Check that %s holds the token of a tunnel that still exists." % secret_name)
            return None
    print("Tunnel not registered yet; it keeps trying in the background. Last log lines:")
    print("\n".join(new_log().splitlines()[-15:]))
    return proc


def keep_alive(port, names, extra=None, interval=60):
    """The last cell: prints one status line a minute so the Kaggle session stays active.
    Interrupting it only stops the printing; the app keeps running."""
    try:
        while True:
            try:
                urllib.request.urlopen("http://127.0.0.1:%d/health" % port, timeout=5)
                parts = ["ui=up"]
            except Exception:
                parts = ["ui=DOWN"]
            for n in names:
                if n == "cloudflared" and not os.path.exists(_pidfile(n)):
                    parts.append("tunnel=off")
                else:
                    parts.append("%s=%s" % (n, "up" if process_alive(n) else "DOWN"))
            gpus = gpu_info()
            if gpus:
                parts.append("GPU " + ", ".join("%d: %.1f/%.0f GB %d%%" % (g["index"], g["mem_used_mb"] / 1024,
                                                                           g["mem_total_mb"] / 1024, g["util"])
                                                for g in gpus))
            if extra:
                try:
                    s = extra()
                    if s:
                        parts.append(s)
                except Exception as e:
                    parts.append("status error: %s" % e)
            print(time.strftime("%H:%M:%S"), " ".join(parts), flush=True)
            time.sleep(interval)
    except KeyboardInterrupt:
        print("Stopped watching. Everything keeps running in the background; re-run this cell to watch again.")


def api_get(port, path, password, timeout=10):
    """GET a JSON endpoint of a local app (used by keep-alive status lines)."""
    req = urllib.request.Request("http://127.0.0.1:%d%s" % (port, path),
                                 headers={"Authorization": "Bearer %s" % password} if password else {})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.loads(r.read().decode("utf-8"))
''')
print("wrote", os.path.join(APP_DIR, 'studio_http.py'))

In [ ]:
# Shared page styles
# Writes kit.css into APP_DIR. Safe to re-run.
import os
APP_DIR = '/kaggle/working/comfy_app'
os.makedirs(APP_DIR, exist_ok=True)
with open(os.path.join(APP_DIR, 'kit.css'), "w", encoding="utf-8") as f:
    f.write(r'''/* Shared look for the agentic-notebooks app pages (served at /static/kit.css). */
:root{--bg:#f5f6f8;--panel:#fff;--panel2:#f0f2f5;--text:#1b1f24;--muted:#626b76;--line:#dde1e6;
  --accent:#2f6fec;--accent-ink:#fff;--ok:#1a7f4b;--warn:#a96500;--bad:#c4372f;--radius:10px;
  --mono:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace}
@media (prefers-color-scheme:dark){:root{--bg:#0f1114;--panel:#171a1f;--panel2:#1e2228;--text:#e7e9ec;
  --muted:#9aa3ad;--line:#2b3038;--accent:#5b8ff5;--ok:#3fbf7f;--warn:#e0a03a;--bad:#ef6a60}}
*{box-sizing:border-box}
html,body{margin:0;background:var(--bg);color:var(--text);font:14px/1.5 system-ui,-apple-system,"Segoe UI",Roboto,sans-serif}
a{color:var(--accent)}
header.top{display:flex;align-items:center;gap:16px;flex-wrap:wrap;padding:12px 20px;border-bottom:1px solid var(--line);background:var(--panel)}
header.top h1{font-size:17px;margin:0;font-weight:650}
header.top .sub{color:var(--muted);font-size:13px}
header.top .right{margin-left:auto;display:flex;gap:10px;align-items:center;flex-wrap:wrap}
nav.tabs{display:flex;gap:4px;padding:0 20px;border-bottom:1px solid var(--line);background:var(--panel);overflow-x:auto}
nav.tabs button{border:0;background:none;color:var(--muted);padding:10px 12px;font:inherit;cursor:pointer;border-bottom:2px solid transparent;white-space:nowrap}
nav.tabs button.on{color:var(--text);border-bottom-color:var(--accent);font-weight:600}
main{max-width:1180px;margin:0 auto;padding:20px}
[data-pane]{display:none}[data-pane].on{display:block}
.card{background:var(--panel);border:1px solid var(--line);border-radius:var(--radius);padding:16px;margin-bottom:16px}
.card h2{font-size:15px;margin:0 0 12px}
.card h3{font-size:14px;margin:14px 0 8px}
.grid{display:grid;gap:16px;grid-template-columns:repeat(auto-fit,minmax(280px,1fr))}
.row{display:flex;gap:10px;align-items:center;flex-wrap:wrap}
.col{display:flex;flex-direction:column;gap:6px}
.muted{color:var(--muted)}.small{font-size:12px}.mono{font-family:var(--mono);font-size:12.5px}
label{display:flex;flex-direction:column;gap:4px;font-size:12.5px;color:var(--muted)}
label.inline{flex-direction:row;align-items:center;gap:6px;color:var(--text);font-size:13px}
input,select,textarea{font:inherit;color:var(--text);background:var(--panel2);border:1px solid var(--line);border-radius:8px;padding:7px 9px;min-width:0}
input[type=checkbox]{width:auto}
textarea{width:100%;resize:vertical;min-height:90px}
input:focus,select:focus,textarea:focus{outline:2px solid color-mix(in srgb,var(--accent) 40%,transparent);border-color:var(--accent)}
button,.btn{font:inherit;border:1px solid var(--line);background:var(--panel2);color:var(--text);border-radius:8px;padding:7px 13px;cursor:pointer;text-decoration:none;display:inline-flex;align-items:center;gap:6px}
button.primary,.btn.primary{background:var(--accent);border-color:var(--accent);color:var(--accent-ink);font-weight:600}
button.danger{color:var(--bad)}
button:disabled{opacity:.5;cursor:default}
table{width:100%;border-collapse:collapse}
th,td{text-align:left;padding:7px 8px;border-bottom:1px solid var(--line);vertical-align:top}
th{font-size:12px;color:var(--muted);font-weight:600}
.badge{display:inline-block;padding:1px 8px;border-radius:999px;font-size:12px;background:var(--panel2);border:1px solid var(--line)}
.badge.ok{color:var(--ok)}.badge.warn{color:var(--warn)}.badge.bad{color:var(--bad)}
.bar{height:6px;background:var(--panel2);border-radius:99px;overflow:hidden}
.bar>i{display:block;height:100%;background:var(--accent);transition:width .3s}
pre.log{background:var(--panel2);border:1px solid var(--line);border-radius:8px;padding:10px;max-height:320px;overflow:auto;font:12px/1.45 var(--mono);white-space:pre-wrap;margin:0}
.drop{border:2px dashed var(--line);border-radius:var(--radius);padding:22px;text-align:center;color:var(--muted)}
.drop.over{border-color:var(--accent);color:var(--text)}
#toasts{position:fixed;right:16px;bottom:16px;display:flex;flex-direction:column;gap:8px;z-index:50}
.toast{background:var(--panel);border:1px solid var(--line);border-left:4px solid var(--accent);padding:10px 14px;border-radius:8px;box-shadow:0 6px 20px rgba(0,0,0,.15);max-width:380px}
.toast.bad{border-left-color:var(--bad)}.toast.ok{border-left-color:var(--ok)}
.empty{color:var(--muted);padding:18px;text-align:center}
@media (max-width:640px){main{padding:12px}header.top{padding:10px 12px}nav.tabs{padding:0 8px}}
''')
print("wrote", os.path.join(APP_DIR, 'kit.css'))

In [ ]:
# Shared page scripts
# Writes kit.js into APP_DIR. Safe to re-run.
import os
APP_DIR = '/kaggle/working/comfy_app'
os.makedirs(APP_DIR, exist_ok=True)
with open(os.path.join(APP_DIR, 'kit.js'), "w", encoding="utf-8") as f:
    f.write(r'''// Shared helpers for the agentic-notebooks app pages (served at /static/kit.js).
// The browser already holds the HTTP Basic login, so fetch() calls need no token.
const $ = (sel, root = document) => root.querySelector(sel);
const $$ = (sel, root = document) => [...root.querySelectorAll(sel)];

function el(tag, attrs = {}, ...kids) {
  const e = document.createElement(tag);
  for (const [k, v] of Object.entries(attrs || {})) {
    if (v == null || v === false) continue;
    if (k.startsWith("on")) e.addEventListener(k.slice(2), v);
    else if (k === "html") e.innerHTML = v;
    else if (k === "style" && typeof v === "object") Object.assign(e.style, v);
    else e.setAttribute(k, v === true ? "" : v);
  }
  for (const kid of kids.flat()) if (kid != null && kid !== false) e.append(kid.nodeType ? kid : String(kid));
  return e;
}

async function api(path, opts = {}) {
  const init = { method: opts.method || (opts.json !== undefined || opts.body ? "POST" : "GET"), headers: {} };
  if (opts.json !== undefined) { init.body = JSON.stringify(opts.json); init.headers["Content-Type"] = "application/json"; }
  else if (opts.body) init.body = opts.body;
  const r = await fetch(path, init);
  const ct = r.headers.get("Content-Type") || "";
  const data = ct.includes("json") ? await r.json() : await r.text();
  if (!r.ok) throw new Error((data && data.error) || r.status + " " + r.statusText);
  return data;
}

function toast(msg, kind = "") {
  let box = $("#toasts");
  if (!box) { box = el("div", { id: "toasts" }); document.body.append(box); }
  const t = el("div", { class: "toast " + kind }, msg);
  box.append(t);
  setTimeout(() => t.remove(), kind === "bad" ? 9000 : 4500);
}

async function guard(fn, okMsg) {
  try { const r = await fn(); if (okMsg) toast(okMsg, "ok"); return r; }
  catch (e) { toast(e.message || String(e), "bad"); throw e; }
}

function setupTabs(onChange) {
  const btns = $$("nav.tabs button[data-tab]");
  const show = (name) => {
    btns.forEach(b => b.classList.toggle("on", b.dataset.tab === name));
    $$("[data-pane]").forEach(p => p.classList.toggle("on", p.dataset.pane === name));
    try { history.replaceState(null, "", "#" + name); } catch (e) {}
    if (onChange) onChange(name);
  };
  btns.forEach(b => b.addEventListener("click", () => show(b.dataset.tab)));
  const first = location.hash.slice(1);
  show(btns.some(b => b.dataset.tab === first) ? first : btns[0].dataset.tab);
  return show;
}

function fmtBytes(n) {
  if (n == null) return "";
  const u = ["B", "KB", "MB", "GB", "TB"]; let i = 0;
  while (n >= 1024 && i < u.length - 1) { n /= 1024; i++; }
  return (i ? n.toFixed(1) : n) + " " + u[i];
}

function fmtDur(s) {
  if (s == null || isNaN(s)) return "";
  s = Math.max(0, Math.round(s));
  const h = Math.floor(s / 3600), m = Math.floor(s % 3600 / 60), sec = s % 60;
  return (h ? h + ":" + String(m).padStart(2, "0") : m) + ":" + String(sec).padStart(2, "0");
}

function fmtAgo(ts) {
  if (!ts) return "";
  const d = Date.now() / 1000 - ts;
  if (d < 60) return "just now";
  if (d < 3600) return Math.floor(d / 60) + " min ago";
  if (d < 86400) return Math.floor(d / 3600) + " h ago";
  return new Date(ts * 1000).toLocaleString();
}

function badge(state) {
  const kind = { done: "ok", ready: "ok", running: "warn", queued: "", loading: "warn", error: "bad", stopped: "bad" }[state] || "";
  return el("span", { class: "badge " + kind }, state || "?");
}

function progressBar(frac) {
  return el("div", { class: "bar" }, el("i", { style: { width: Math.round(Math.max(0, Math.min(1, frac || 0)) * 100) + "%" } }));
}

function every(ms, fn) {
  let stopped = false;
  const tick = async () => { if (stopped) return; try { await fn(); } catch (e) {} if (!stopped) setTimeout(tick, ms); };
  tick();
  return () => { stopped = true; };
}

function dropZone(zone, input, onFiles) {
  zone.addEventListener("click", () => input.click());
  input.addEventListener("change", () => { if (input.files.length) onFiles([...input.files]); input.value = ""; });
  zone.addEventListener("dragover", e => { e.preventDefault(); zone.classList.add("over"); });
  zone.addEventListener("dragleave", () => zone.classList.remove("over"));
  zone.addEventListener("drop", e => { e.preventDefault(); zone.classList.remove("over"); if (e.dataTransfer.files.length) onFiles([...e.dataTransfer.files]); });
}

function mcpHelp(name) {
  const url = location.origin + "/mcp";
  return "claude mcp add --transport http " + name + " " + url + " --header \"Authorization: Bearer <password>\"";
}
''')
print("wrote", os.path.join(APP_DIR, 'kit.js'))

In [ ]:
# Gateway on port 7860: password, ComfyUI proxy, /flux page, API and MCP
# Writes comfy_gateway.py into APP_DIR. Safe to re-run.
import os
APP_DIR = '/kaggle/working/comfy_app'
os.makedirs(APP_DIR, exist_ok=True)
with open(os.path.join(APP_DIR, 'comfy_gateway.py'), "w", encoding="utf-8") as f:
    f.write(r'''"""ComfyUI + FLUX gateway on port 7860, the only port the Cloudflare Tunnel exposes.

Runs as its own process, started by the notebook with studio_http.spawn():
    APP_PASSWORD=... COMFY_URL=http://127.0.0.1:8188 python comfy_gateway.py --port 7860

- Everything ComfyUI serves (its node editor at /, its HTTP API and the /ws websocket) is proxied
  behind the password.
- /flux is a one-box prompt -> image page.
- /flux/api/* is a small JSON API, and /mcp offers the same features to agents as MCP tools.
"""
import argparse
import io
import json
import os
import random
import time
import urllib.error
import urllib.request
import uuid
from urllib.parse import quote, urlencode

import studio_http as K

HERE = os.path.dirname(os.path.abspath(__file__))
COMFY = os.environ.get("COMFY_URL", "http://127.0.0.1:8188").rstrip("/")
VARIANT = os.environ.get("FLUX_VARIANT", "schnell")
CKPT = os.environ.get("FLUX_CKPT", "flux1-schnell-fp8.safetensors")
DEFAULT_STEPS = {"schnell": 4, "dev": 20}.get(VARIANT, 20)
CLIENT_ID = "flux-gateway-" + uuid.uuid4().hex[:8]
MAX_WAIT = 85                     # Cloudflare ends requests after 100 s
log = K.file_logger(os.environ.get("APP_LOG", "/kaggle/working/logs/gateway.log"), "gateway")
JOBS = {}                         # prompt_id -> what we submitted (newest last)


# ====================================================================== ComfyUI client
def _comfy_error(detail):
    if not isinstance(detail, dict):
        return str(detail)[:500]
    parts = []
    err = detail.get("error")
    if isinstance(err, dict):
        parts.append(err.get("message") or err.get("type") or "error")
        if err.get("details"):
            parts.append(str(err["details"]))
    for node, info in (detail.get("node_errors") or {}).items():
        for e in info.get("errors", []):
            parts.append("node %s (%s): %s %s" % (node, info.get("class_type", "?"), e.get("message", ""),
                                                  e.get("details", "")))
    return "; ".join(p for p in parts if p) or json.dumps(detail)[:500]


def comfy(path, data=None, timeout=30, raw=False):
    req = urllib.request.Request(COMFY + path, data=None if data is None else json.dumps(data).encode(),
                                 headers={"Content-Type": "application/json"} if data is not None else {})
    try:
        with urllib.request.urlopen(req, timeout=timeout) as r:
            body = r.read()
    except urllib.error.HTTPError as e:
        body = e.read()
        try:
            detail = json.loads(body)
        except ValueError:
            detail = body.decode("utf-8", "replace")
        raise K.HTTPError(400 if e.code < 500 else 502, "ComfyUI: " + _comfy_error(detail))
    except OSError as e:
        raise K.HTTPError(502, "ComfyUI is not reachable (%s). Check the ComfyUI cell and "
                               "/kaggle/working/logs/comfyui.log" % type(e).__name__)
    if raw:
        return body
    return json.loads(body) if body.strip() else {}


def view_url(img):
    return "/view?" + urlencode({"filename": img["filename"], "subfolder": img.get("subfolder", ""),
                                 "type": img.get("type", "output")})


# ====================================================================== workflows
def _int(v, default, lo, hi, name):
    v = default if v in (None, "") else int(v)
    if not lo <= v <= hi:
        raise ValueError("%s must be between %d and %d" % (name, lo, hi))
    return v


def flux_workflow(prompt, width=1024, height=1024, steps=None, seed=None, guidance=3.5, batch_size=1,
                  prefix="flux/api"):
    prompt = str(prompt or "").strip()
    if not prompt:
        raise ValueError("prompt is empty")
    if len(prompt) > 4000:
        raise ValueError("prompt is longer than 4000 characters")
    width = _int(width, 1024, 256, 2048, "width") // 16 * 16
    height = _int(height, 1024, 256, 2048, "height") // 16 * 16
    steps = _int(steps, DEFAULT_STEPS, 1, 60, "steps")
    batch_size = _int(batch_size, 1, 1, 4, "batch_size")
    seed = random.randint(0, 2 ** 48) if seed in (None, "", -1) else int(seed)
    positive = ["6", 0]
    wf = {
        "4": {"class_type": "CheckpointLoaderSimple", "inputs": {"ckpt_name": CKPT}},
        "6": {"class_type": "CLIPTextEncode", "inputs": {"text": prompt, "clip": ["4", 1]}},
        "33": {"class_type": "CLIPTextEncode", "inputs": {"text": "", "clip": ["4", 1]}},
        "27": {"class_type": "EmptySD3LatentImage", "inputs": {"width": width, "height": height,
                                                               "batch_size": batch_size}},
        "31": {"class_type": "KSampler", "inputs": {
            "seed": seed, "steps": steps, "cfg": 1.0, "sampler_name": "euler", "scheduler": "simple",
            "denoise": 1.0, "model": ["4", 0], "positive": positive, "negative": ["33", 0],
            "latent_image": ["27", 0]}},
        "8": {"class_type": "VAEDecode", "inputs": {"samples": ["31", 0], "vae": ["4", 2]}},
        "9": {"class_type": "SaveImage", "inputs": {"filename_prefix": prefix, "images": ["8", 0]}},
    }
    if VARIANT == "dev":   # FLUX.1-dev uses distilled guidance instead of CFG
        wf["35"] = {"class_type": "FluxGuidance", "inputs": {"guidance": float(guidance or 3.5),
                                                             "conditioning": ["6", 0]}}
        wf["31"]["inputs"]["positive"] = ["35", 0]
    params = {"prompt": prompt, "width": width, "height": height, "steps": steps, "seed": seed,
              "batch_size": batch_size, "guidance": float(guidance or 3.5) if VARIANT == "dev" else None}
    return wf, params


def submit(workflow, kind, params=None):
    if not isinstance(workflow, dict) or not workflow:
        raise ValueError("workflow must be a ComfyUI API-format object: {node_id: {class_type, inputs}}")
    if "nodes" in workflow and "links" in workflow:
        raise ValueError("this is a UI-format workflow; in ComfyUI use Workflow -> Export (API) instead")
    r = comfy("/prompt", {"prompt": workflow, "client_id": CLIENT_ID})
    pid = r.get("prompt_id")
    if not pid:
        raise K.HTTPError(502, "ComfyUI did not return a prompt_id: %s" % _comfy_error(r))
    JOBS[pid] = {"job_id": pid, "kind": kind, "params": params or {}, "created": time.time()}
    while len(JOBS) > 200:
        JOBS.pop(next(iter(JOBS)))
    log.info("queued %s job %s", kind, pid)
    return pid


def _queue():
    q = comfy("/queue")
    running = [x[1] for x in q.get("queue_running", [])]
    pending = [x[1] for x in sorted(q.get("queue_pending", []), key=lambda x: x[0])]
    return running, pending


def job_status(pid, history=None, queue=None):
    out = dict(JOBS.get(pid) or {"job_id": pid})
    h = (history if history is not None else comfy("/history/" + quote(pid))).get(pid)
    if h:
        st = h.get("status") or {}
        images = []
        for node_out in (h.get("outputs") or {}).values():
            for img in node_out.get("images", []):
                if img.get("type") == "output":
                    images.append({"filename": img["filename"], "subfolder": img.get("subfolder", ""),
                                   "url": view_url(img)})
        if st.get("status_str") == "error":
            msg = "failed"
            for name, data in st.get("messages", []):
                if name == "execution_error":
                    msg = "%s (node %s %s)" % (data.get("exception_message", "").strip(), data.get("node_id"),
                                               data.get("node_type"))
                elif name == "execution_interrupted":
                    msg = "cancelled"
            out.update(state="error", error=msg, images=images)
        else:
            out.update(state="done", images=images)
        times = [m[1].get("timestamp") for m in st.get("messages", []) if m[0] in ("execution_start",
                                                                                   "execution_success")]
        if len(times) == 2 and all(times):
            out["seconds"] = round((times[1] - times[0]) / 1000, 1)
        return out
    running, pending = queue or _queue()
    if pid in running:
        out.update(state="running")
    elif pid in pending:
        out.update(state="queued", queue_position=pending.index(pid) + 1)
    else:
        out.update(state="unknown", error="ComfyUI has no job with this id (it may have restarted)")
    return out


def wait_job(pid, seconds):
    seconds = max(0, min(MAX_WAIT, float(seconds or 0)))
    t = time.time()
    while True:
        st = job_status(pid)
        if st["state"] in ("done", "error", "unknown") or time.time() - t >= seconds:
            return st
        time.sleep(1.0)


def image_bytes(img, max_side=1024):
    """Image as JPEG for MCP replies (PNGs from FLUX are often 1.5 MB+)."""
    data = comfy(view_url(dict(img, type="output")), raw=True, timeout=60)
    try:
        from PIL import Image
        im = Image.open(io.BytesIO(data)).convert("RGB")
        im.thumbnail((max_side, max_side))
        buf = io.BytesIO()
        im.save(buf, "JPEG", quality=90)
        return buf.getvalue(), "image/jpeg"
    except Exception:
        return data, "image/png"


def tool_reply(st, return_image=True):
    blocks = [K.text_block(json.dumps(st, indent=1, default=str))]
    if st.get("state") == "done" and return_image:
        for img in st.get("images", [])[:4]:
            try:
                data, mime = image_bytes(img)
                blocks.append(K.image_block(data, mime))
            except Exception as e:
                blocks.append(K.text_block("could not attach %s: %s" % (img["filename"], e)))
    elif st.get("state") in ("queued", "running"):
        blocks.append(K.text_block("Still working. Call get_job with this job_id (wait_seconds up to 85)."))
    return K.ToolContent(blocks)


def models():
    def options(node, field):
        try:
            info = comfy("/object_info/" + node)[node]["input"]["required"][field][0]
            return info if isinstance(info, list) else []
        except (K.HTTPError, KeyError, IndexError, TypeError):
            return []
    return {"checkpoints": options("CheckpointLoaderSimple", "ckpt_name"),
            "loras": options("LoraLoader", "lora_name"),
            "default_checkpoint": CKPT, "variant": VARIANT}


def info():
    out = {"variant": VARIANT, "checkpoint": CKPT, "default_steps": DEFAULT_STEPS,
           "guidance": VARIANT == "dev"}
    try:
        running, pending = _queue()
        out["queue"] = {"running": len(running), "pending": len(pending)}
        stats = comfy("/system_stats")
        out["devices"] = [{"name": d.get("name"), "vram_free_gb": round(d.get("vram_free", 0) / 2 ** 30, 1),
                           "vram_total_gb": round(d.get("vram_total", 0) / 2 ** 30, 1)}
                          for d in stats.get("devices", [])]
        out["comfyui"] = (stats.get("system") or {}).get("comfyui_version")
        out["ready"] = True
    except K.HTTPError as e:
        out.update(ready=False, error=str(e))
    return out


def recent_jobs(limit=30):
    hist = comfy("/history?max_items=%d" % limit)
    queue = _queue()
    ids = list(dict.fromkeys(list(reversed(list(JOBS))) + queue[0] + queue[1] + list(reversed(list(hist)))))
    return [job_status(pid, hist, queue) for pid in ids[:limit]]


def cancel(pid):
    running, pending = _queue()
    if pid in running:
        comfy("/interrupt", {})
        return {"job_id": pid, "cancelled": "interrupted the running job"}
    if pid in pending:
        comfy("/queue", {"delete": [pid]})
        return {"job_id": pid, "cancelled": "removed from the queue"}
    return {"job_id": pid, "cancelled": False, "reason": "job is not queued or running"}


# ====================================================================== app
def build_app():
    app = K.App("ComfyUI FLUX Studio", password=os.environ.get("APP_PASSWORD"), proxy=COMFY, log=log,
                max_body=200 << 20, instructions=(
                    "Text-to-image with FLUX.1-%s on ComfyUI. generate_image queues a job and waits up to "
                    "wait_seconds (max 85) for it; if the reply says queued/running, call get_job with the "
                    "job_id. run_workflow accepts any ComfyUI API-format workflow. Images are also downloadable "
                    "from the url fields (relative to this server, same password)." % VARIANT))

    app.page("/flux", os.path.join(HERE, "flux_ui.html"))
    app.page("/flux/", os.path.join(HERE, "flux_ui.html"))
    app.static("/flux/static/", HERE)

    @app.route("GET", "/flux/api/info")
    def _info(req):
        return info()

    @app.route("GET", "/flux/api/models")
    def _models(req):
        return models()

    @app.route("POST", "/flux/api/generate")
    def _generate(req):
        b = req.json()
        wf, params = flux_workflow(b.get("prompt"), b.get("width"), b.get("height"), b.get("steps"), b.get("seed"),
                                   b.get("guidance"), b.get("batch_size"), prefix="flux/web")
        return {"job_id": submit(wf, "generate", params), "params": params}

    @app.route("POST", "/flux/api/workflow")
    def _workflow(req):
        b = req.json()
        return {"job_id": submit(b.get("workflow", b), "workflow")}

    @app.route("GET", "/flux/api/jobs")
    def _jobs(req):
        return {"jobs": recent_jobs(req.arg("limit", 30, int))}

    @app.route("GET", r"/flux/api/jobs/(?P<pid>[\w-]+)")
    def _job(req):
        return wait_job(req.params["pid"], req.arg("wait", 0, float))

    @app.route("POST", r"/flux/api/jobs/(?P<pid>[\w-]+)/cancel")
    def _cancel(req):
        return cancel(req.params["pid"])

    # ---------------------------------------------------------------- MCP tools
    @app.tool("generate_image", "Generate images from a text prompt with FLUX.1-%s. Returns the job status, and "
              "the images inline once done (JPEG previews; full PNGs at the url fields)." % VARIANT, {
                  "prompt": {"type": "string", "description": "What to draw. FLUX follows long, literal prompts well."},
                  "width": {"type": "integer", "default": 1024, "minimum": 256, "maximum": 2048},
                  "height": {"type": "integer", "default": 1024, "minimum": 256, "maximum": 2048},
                  "steps": {"type": "integer", "description": "Default %d" % DEFAULT_STEPS},
                  "seed": {"type": "integer", "description": "Omit for random"},
                  "guidance": {"type": "number", "default": 3.5, "description": "FLUX.1-dev only"},
                  "batch_size": {"type": "integer", "default": 1, "minimum": 1, "maximum": 4},
                  "wait_seconds": {"type": "number", "default": 75, "maximum": 85},
                  "return_image": {"type": "boolean", "default": True}}, ["prompt"])
    def t_generate(prompt, width=1024, height=1024, steps=None, seed=None, guidance=3.5, batch_size=1,
                   wait_seconds=75, return_image=True):
        wf, params = flux_workflow(prompt, width, height, steps, seed, guidance, batch_size, prefix="flux/mcp")
        return tool_reply(wait_job(submit(wf, "generate", params), wait_seconds), return_image)

    @app.tool("get_job", "Status of a job; returns its images once done.", {
        "job_id": {"type": "string"},
        "wait_seconds": {"type": "number", "default": 0, "maximum": 85},
        "return_image": {"type": "boolean", "default": True}}, ["job_id"])
    def t_get_job(job_id, wait_seconds=0, return_image=True):
        return tool_reply(wait_job(str(job_id), wait_seconds), return_image)

    @app.tool("run_workflow", "Queue any ComfyUI workflow in API format ({node_id: {class_type, inputs}}; in "
              "ComfyUI: Workflow -> Export (API)). Use list_models for valid checkpoint and LoRA names.", {
                  "workflow": {"type": "object"},
                  "wait_seconds": {"type": "number", "default": 75, "maximum": 85},
                  "return_image": {"type": "boolean", "default": True}}, ["workflow"])
    def t_workflow(workflow, wait_seconds=75, return_image=True):
        return tool_reply(wait_job(submit(workflow, "workflow"), wait_seconds), return_image)

    @app.tool("list_jobs", "Recent jobs, newest first.", {"limit": {"type": "integer", "default": 20}})
    def t_jobs(limit=20):
        return [{k: v for k, v in j.items() if k != "params"} for j in recent_jobs(min(int(limit), 100))]

    @app.tool("cancel_job", "Cancel a queued or running job.", {"job_id": {"type": "string"}}, ["job_id"])
    def t_cancel(job_id):
        return cancel(str(job_id))

    @app.tool("list_models", "Checkpoints and LoRAs ComfyUI can load.")
    def t_models():
        return models()

    @app.tool("server_status", "Queue length, GPU memory and the loaded FLUX variant.")
    def t_status():
        return info()

    return app


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--port", type=int, default=7860)
    build_app().serve_forever(ap.parse_args().port)
''')
print("wrote", os.path.join(APP_DIR, 'comfy_gateway.py'))

In [ ]:
# The /flux page
# Writes flux_ui.html into APP_DIR. Safe to re-run.
import os
APP_DIR = '/kaggle/working/comfy_app'
os.makedirs(APP_DIR, exist_ok=True)
with open(os.path.join(APP_DIR, 'flux_ui.html'), "w", encoding="utf-8") as f:
    f.write(r'''<!doctype html>
<html lang="en"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>FLUX Studio</title>
<link rel="stylesheet" href="/flux/static/kit.css">
<style>
.gallery{display:grid;gap:12px;grid-template-columns:repeat(auto-fill,minmax(220px,1fr))}
.gallery figure{margin:0;background:var(--panel2);border:1px solid var(--line);border-radius:10px;overflow:hidden}
.gallery img{width:100%;display:block;aspect-ratio:1;object-fit:cover;cursor:zoom-in}
.gallery figcaption{padding:8px 10px;font-size:12px;color:var(--muted);display:flex;justify-content:space-between;gap:8px}
.gallery .ph{aspect-ratio:1;display:flex;align-items:center;justify-content:center;flex-direction:column;gap:8px;color:var(--muted);padding:14px;text-align:center}
#prompt{min-height:110px}
dialog{border:0;padding:0;background:transparent;max-width:95vw}
dialog img{max-width:95vw;max-height:90vh;display:block;border-radius:8px}
dialog::backdrop{background:rgba(0,0,0,.75)}
</style></head><body>
<header class="top">
  <h1>FLUX Studio</h1><span class="sub" id="variant"></span>
  <div class="right"><span class="small muted" id="status">connecting…</span><a class="btn" href="/">Open ComfyUI editor</a></div>
</header>
<main>
  <div class="card">
    <form id="f" class="col">
      <textarea id="prompt" placeholder="A cosy reading nook in a treehouse at dusk, warm lantern light, rain on the window, detailed illustration"></textarea>
      <div class="row">
        <label>Size<select id="size">
          <option value="1024x1024">1024 × 1024 square</option>
          <option value="832x1216">832 × 1216 portrait</option>
          <option value="1216x832">1216 × 832 landscape</option>
          <option value="768x1344">768 × 1344 phone</option>
          <option value="768x768">768 × 768 fast</option>
        </select></label>
        <label>Steps<input id="steps" type="number" min="1" max="60" style="width:80px"></label>
        <label id="gwrap" hidden>Guidance<input id="guidance" type="number" step="0.1" value="3.5" style="width:80px"></label>
        <label>Images<select id="batch"><option>1</option><option>2</option><option>4</option></select></label>
        <label>Seed<input id="seed" placeholder="random" style="width:130px"></label>
        <button class="primary" id="go" style="margin-left:auto;align-self:flex-end">Generate</button>
      </div>
    </form>
  </div>
  <div class="card">
    <div class="row" style="margin-bottom:10px"><h2 style="margin:0">Recent images</h2><span class="small muted">Saved in /kaggle/working/comfyui-output; click to enlarge.</span></div>
    <div class="gallery" id="gallery"><div class="empty">No images yet.</div></div>
  </div>
  <div class="card">
    <h2>For agents</h2>
    <p class="small muted">MCP endpoint (tools: generate_image, get_job, run_workflow, list_jobs, cancel_job, list_models, server_status):</p>
    <pre class="log" id="mcp"></pre>
  </div>
</main>
<dialog id="dlg"><img id="big" alt=""></dialog>
<script src="/flux/static/kit.js"></script>
<script>
let defaults = {steps: 4};
const pending = new Set();
$("#mcp").textContent = mcpHelp("flux");

async function loadInfo() {
  const i = await api("/flux/api/info");
  defaults.steps = i.default_steps;
  $("#variant").textContent = "FLUX.1-" + i.variant + " · " + i.checkpoint;
  if (!$("#steps").value) $("#steps").value = i.default_steps;
  $("#gwrap").hidden = !i.guidance;
  if (!i.ready) { $("#status").textContent = "ComfyUI not ready: " + i.error; return; }
  const d = (i.devices || [])[0];
  $("#status").textContent = "queue " + i.queue.running + " running, " + i.queue.pending + " waiting" +
    (d ? " · " + d.vram_free_gb + "/" + d.vram_total_gb + " GB free" : "");
}

function figure(job, img) {
  const p = job.params || {};
  const cap = el("figcaption", {}, el("span", {}, p.seed != null ? "seed " + p.seed : job.kind || ""),
                 el("a", {href: img.url, download: img.filename}, "download"));
  const im = el("img", {src: img.url, loading: "lazy", alt: p.prompt || "", title: p.prompt || "",
                        onclick: () => { $("#big").src = img.url; $("#dlg").showModal(); }});
  return el("figure", {}, im, cap);
}

function placeholder(job) {
  const txt = job.state === "queued" ? "Waiting (#" + job.queue_position + " in queue)" :
              job.state === "running" ? "Generating… the first image also loads the model (1–3 min)" :
              job.state === "error" ? "Failed: " + job.error : job.state;
  return el("figure", {}, el("div", {class: "ph"}, badge(job.state), el("div", {class: "small"}, txt),
            el("div", {class: "small"}, (job.params || {}).prompt || "")));
}

async function loadJobs() {
  const {jobs} = await api("/flux/api/jobs?limit=40");
  const g = $("#gallery");
  g.replaceChildren();
  for (const j of jobs) {
    if (j.state === "done") (j.images || []).forEach(img => g.append(figure(j, img)));
    else if (j.state !== "unknown") g.append(placeholder(j));
  }
  if (!g.children.length) g.append(el("div", {class: "empty"}, "No images yet."));
  return jobs.some(j => j.state === "queued" || j.state === "running");
}

$("#f").addEventListener("submit", async e => {
  e.preventDefault();
  const [w, h] = $("#size").value.split("x").map(Number);
  const seed = $("#seed").value.trim();
  const body = {prompt: $("#prompt").value, width: w, height: h, steps: +$("#steps").value || defaults.steps,
                batch_size: +$("#batch").value, guidance: +$("#guidance").value};
  if (seed) body.seed = +seed;
  $("#go").disabled = true;
  try { await guard(() => api("/flux/api/generate", {json: body}), "Queued"); await loadJobs(); }
  finally { $("#go").disabled = false; }
});
$("#prompt").addEventListener("keydown", e => { if (e.key === "Enter" && (e.ctrlKey || e.metaKey)) $("#f").requestSubmit(); });
$("#dlg").addEventListener("click", () => $("#dlg").close());

every(5000, loadInfo);
every(3000, loadJobs);
</script></body></html>
''')
print("wrote", os.path.join(APP_DIR, 'flux_ui.html'))

In [ ]:
# 3. Load the helpers
import os, sys
APP_DIR = '/kaggle/working/comfy_app'
_missing = [f for f in ("studio_http.py", "kit.css", "kit.js") if not os.path.exists(os.path.join(APP_DIR, f))]
if _missing:
    raise RuntimeError("Run the 'Writes ... into APP_DIR' cells above first (missing: %s)" % ", ".join(_missing))
if APP_DIR not in sys.path:
    sys.path.insert(0, APP_DIR)
import studio_http as K   # Notebook helpers: kaggle_secret, spawn, wait_http, tail, start_tunnel, keep_alive

In [ ]:
# 4. Install ComfyUI and cloudflared (quiet; about 3 minutes; re-running skips finished steps)
import re, subprocess, shutil
def pip_install(*args, keep_core=True):
    """pip install, quietly. keep_core pins Kaggle's own torch / numpy builds so nothing replaces them."""
    cons = "/tmp/keep_core.txt"
    lines = []
    if keep_core:
        import importlib.metadata as md
        for pkg in ("torch", "torchvision", "torchaudio", "numpy"):
            try:
                lines.append("%s==%s" % (pkg, md.version(pkg)))
            except md.PackageNotFoundError:
                pass
    open(cons, "w").write("\n".join(lines) + "\n")
    cmd = [sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", "-c", cons, *args]
    p = subprocess.run(cmd, capture_output=True, text=True)
    if p.returncode != 0:
        print(p.stdout[-2000:], p.stderr[-3000:])
        raise RuntimeError("pip install failed: " + " ".join(args))

def sh(cmd):
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if p.returncode != 0:
        print(p.stdout[-2000:], p.stderr[-3000:])
        raise RuntimeError("failed: " + cmd[:120])
    return p.stdout

os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
if not os.path.isdir(COMFY_DIR + "/.git"):
    print("ComfyUI source...")
    sh("git clone -q https://github.com/comfyanonymous/ComfyUI %s" % COMFY_DIR)
if COMFYUI_REF:
    sh("cd %s && git fetch -q origin %s && git checkout -q %s" % (COMFY_DIR, COMFYUI_REF, COMFYUI_REF))
print("ComfyUI at", sh("cd %s && git describe --tags --always" % COMFY_DIR).strip())

print("Python packages (keeping Kaggle's own PyTorch)...")
reqs = [l.strip() for l in open(COMFY_DIR + "/requirements.txt") if l.strip() and not l.lstrip().startswith("#")]
keep = [r for r in reqs if re.split(r"[<>=~!\[; ]", r, 1)[0].lower() not in ("torch", "torchvision", "torchaudio")]
open("/tmp/comfy_requirements.txt", "w").write("\n".join(keep) + "\n")
pip_install("-r", "/tmp/comfy_requirements.txt")
pip_install("huggingface_hub")
K.ensure_cloudflared()

import torch
print("Install complete. torch", torch.__version__, "| GPUs:",
      [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())] or "none - pick a GPU accelerator")

In [ ]:
# 5. Download the FLUX checkpoint (~17 GB, a few minutes; skipped if already there)
from huggingface_hub import hf_hub_download

repo, CKPT = CHECKPOINTS[FLUX_VARIANT]
ckpt_dir = os.path.join(COMFY_DIR, "models", "checkpoints")
os.makedirs(ckpt_dir, exist_ok=True)
if os.path.exists(os.path.join(ckpt_dir, CKPT)):
    print("Already downloaded:", CKPT)
else:
    free_gb = shutil.disk_usage(ckpt_dir).free / 2**30
    if free_gb < 20:
        print("Warning: only %.0f GB free under %s; the download needs about 18 GB." % (free_gb, ckpt_dir))
    print("Downloading %s from %s ..." % (CKPT, repo))
    try:
        hf_hub_download(repo, CKPT, local_dir=ckpt_dir, token=K.kaggle_secret("HF_TOKEN"))
    except Exception as e:
        hint = (" FLUX.1-dev is gated: accept its licence on huggingface.co with the account of your HF_TOKEN "
                "secret, and attach that secret." if FLUX_VARIANT == "dev" else "")
        raise RuntimeError("Download failed: %s.%s" % (e, hint)) from None
print("Checkpoint ready:", CKPT)

In [ ]:
# 6. Start ComfyUI in the background on 127.0.0.1:8188 (GPU 0). Log: /kaggle/working/logs/comfyui.log
comfy_env = dict(os.environ, CUDA_VISIBLE_DEVICES="0", PYTHONUNBUFFERED="1")
comfy_cmd = [sys.executable, "main.py", "--listen", "127.0.0.1", "--port", str(COMFY_PORT),
             "--output-directory", OUTPUT_DIR, "--disable-auto-launch", "--preview-method", "auto"] + EXTRA_COMFY_ARGS
K.spawn("comfyui", comfy_cmd, LOG_DIR + "/comfyui.log", env=comfy_env, cwd=COMFY_DIR)
print("Starting ComfyUI...")
if not K.wait_http("http://127.0.0.1:%d/system_stats" % COMFY_PORT, timeout=300, name="comfyui"):
    print(K.tail(LOG_DIR + "/comfyui.log", 40))
    raise RuntimeError("ComfyUI did not start; log above")
print("ComfyUI is up.")

In [ ]:
# 7. Start the gateway on port 7860 (password, /flux page, API, MCP). Log: /kaggle/working/logs/gateway.log
CKPT = CHECKPOINTS[FLUX_VARIANT][1]
PASSWORD = K.ui_password("COMFYUI_UI_PASSWORD")
gw_env = dict(os.environ, APP_PASSWORD=PASSWORD, COMFY_URL="http://127.0.0.1:%d" % COMFY_PORT,
              FLUX_VARIANT=FLUX_VARIANT, FLUX_CKPT=CKPT, APP_LOG=LOG_DIR + "/gateway.log")
K.spawn("gateway", [sys.executable, os.path.join(APP_DIR, "comfy_gateway.py"), "--port", str(PORT)],
        LOG_DIR + "/gateway.log", env=gw_env, cwd=APP_DIR)
if not K.wait_http("http://127.0.0.1:%d/health" % PORT, timeout=30, name="gateway"):
    print(K.tail(LOG_DIR + "/gateway.log", 40))
    raise RuntimeError("The gateway did not start; log above")
for host in ("127.0.0.1", "[::1]"):
    print(host, "->", "up" if K.wait_http("http://%s:%d/health" % (host, PORT), timeout=3) else "not reachable")
print("Paths: / = ComfyUI editor, /flux = quick generator, /mcp = MCP server for agents.")

In [ ]:
# 8. Publish through your Cloudflare Tunnel (token from the COMFYUI_TUNNEL_TOKEN secret, never printed)
tunnel = K.start_tunnel("COMFYUI_TUNNEL_TOKEN", PORT, LOG_DIR + "/cloudflared.log")

In [ ]:
# 9. Keep-alive monitor. Leave it running; interrupting it only stops the status lines.
def queue_line():
    q = K.api_get(PORT, "/flux/api/info", PASSWORD).get("queue") or {}
    return "queue=%s running/%s waiting" % (q.get("running", "?"), q.get("pending", "?"))

K.keep_alive(PORT, ["comfyui", "gateway", "cloudflared"], extra=queue_line)